### Chunking Preprocessed Text for Analysis

In [1]:
# ── CELL 5: Load + Chunk Papers ────────────
import json
import os

# Load saved papers
with open('../data/raw/alzheimer_papers.json') as f:
    papers = json.load(f)

print(f"✅ Loaded: {len(papers):,} papers")

def chunk_abstract(paper, 
                   chunk_size=150,
                   overlap=30):
    """
    Split abstract into overlapping chunks
    
    chunk_size = words per chunk
    overlap    = shared words between chunks
    """
    words    = paper['abstract'].split()
    chunks   = []
    start    = 0

    while start < len(words):
        end   = start + chunk_size
        chunk = " ".join(words[start:end])

        chunks.append({
            "chunk_id": f"{paper['pmid']}_{len(chunks)}",
            "pmid":     paper['pmid'],
            "title":    paper['title'],
            "authors":  paper['authors'],
            "year":     paper['year'],
            "journal":  paper['journal'],
            "keywords": paper['keywords'],
            "chunk":    chunk,
            "source":   "PubMed"
        })

        start += chunk_size - overlap
        if start >= len(words):
            break

    return chunks

# ── Chunk all papers ───────────────────────
all_chunks = []

for paper in papers:
    chunks = chunk_abstract(paper)
    all_chunks.extend(chunks)

# ── Save chunks ────────────────────────────
os.makedirs('../data/processed', exist_ok=True)
CHUNKS_FILE = '../data/processed/chunks.json'

with open(CHUNKS_FILE, 'w') as f:
    json.dump(all_chunks, f, indent=2)

# ── Stats ──────────────────────────────────
print(f"\n{'='*50}")
print(f"📊 CHUNKING REPORT")
print(f"{'='*50}")
print(f"   Papers:          {len(papers):,}")
print(f"   Total chunks:    {len(all_chunks):,}")
print(f"   Avg chunks/paper:{len(all_chunks)//len(papers)}")
print(f"   Chunk size:      150 words")
print(f"   Overlap:         30 words")
print(f"   Saved to:        {CHUNKS_FILE}")
print(f"{'='*50}")
print(f"✅ Chunking complete!")


✅ Loaded: 16,281 papers

📊 CHUNKING REPORT
   Papers:          16,281
   Total chunks:    40,995
   Avg chunks/paper:2
   Chunk size:      150 words
   Overlap:         30 words
   Saved to:        ../data/processed/chunks.json
✅ Chunking complete!


In [ ]:
# ── CELL 6: Generate Embeddings ────────────
from sentence_transformers import SentenceTransformer
import numpy as np
import os

print("🧠 Loading PubMedBERT model...")
model = SentenceTransformer(
    'neuml/pubmedbert-base-embeddings'
)
print("✅ Model loaded!")

# Extract text only
texts = [c['chunk'] for c in all_chunks]

print(f"\n⏳ Embedding {len(texts):,} chunks...")
print(f"   Model:      PubMedBERT 768-dim")
print(f"   Chunks:     {len(texts):,}")
print(f"   Time:       ~30-45 minutes ⏳")
print(f"   Go get coffee! ☕")

# Generate embeddings
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings complete!")
print(f"   Shape: {embeddings.shape}")
print(f"   Expected: ({len(texts):,}, 768)")
